# Agentic Foley System Deep Dive

This notebook is documentation-first. It explains the full system architecture, runtime flow, and debugging workflow for teammates.

**Audience:** engineers onboarding to this repo.

**Scope:** websocket app flow, backend orchestration, VLM perception, planning, execution, scoring, retries, and frontend event rendering.

## 1. System Overview

The project converts either:
- uploaded video -> generated Foley audio + muxed output video, or
- prompt-only input -> generated Foley audio.

Core stack:
- `web/` (Next.js UI): upload/input + live event timeline over WebSocket.
- `server/agent_ws_api.py`: upload endpoint, artifact serving, WebSocket bridge.
- `main.py` (`FoleyOrchestrator`): the agentic loop and final stitching.
- `server/multi_model_api.py`: model endpoints (`/perception`, `/planner`, `/execution`, `/verification`).

## 2. High-Level Request Flow

### 2.1 Video Mode
1. UI uploads file with `PUT /upload-video?filename=...`.
2. UI opens `ws://.../ws/foley` and sends `{action:'start', video_path, prompt}`.
3. `agent_ws_api.py` spins a worker thread and runs `FoleyOrchestrator.run_pipeline`.
4. Orchestrator emits step-by-step events back to UI via callback queue.
5. Final output artifact is served from `/artifacts/video/<name>` and rendered inline in UI.

### 2.2 Prompt-Only Mode
1. UI opens socket with empty `video_path`.
2. `run_audio_only` executes one event through the same inner loop.
3. Output is served from `/artifacts/audio/<name>` and rendered inline in UI.

## 3. Agentic Loop (Core Logic)

For each planned `AudioEvent`:
1. `attempt_started`: choose prompt for this attempt.
2. `execution.generate_audio`: synthesize candidate clip.
3. `verification.evaluate`: CLAP similarity score.
4. `planner.decide_iteration`: controller decides one of:
   - `ACCEPT`
   - `RETRY_REWRITE`
   - `RETRY_BEST`
   - `STOP_BEST`
5. If retrying, loop to next attempt (up to `max_retries`).
6. `event_completed`: select best candidate and score.

After all events:
- overlay event clips on timeline,
- mux audio with video,
- save run trace JSON,
- emit `run_completed`.

## 4. Event Taxonomy (What UI Receives)

### Lifecycle
- `run_started`
- `video_prepared`
- `planning_completed`
- `run_completed`
- `run_failed`

### Perception/VLM (video mode)
- `vlm_keyframes_extracted`
- `vlm_keyframes_downsampled`
- `vlm_request_started`
- `vlm_response_received`
- `vlm_request_failed`
- `perception_completed`

### Iteration loop
- `attempt_started`
- `clap_scored`
- `decision_made`
- `event_completed`

The frontend queues and animates these in sequence to make the loop understandable.

## 5. Why VLM Timelines Can Be Coarse

Even with strict prompting, VLM output quality depends on:
- model capability (small VLMs often produce scene captions),
- number and quality of sampled keyframes,
- token budget (`max_new_tokens`),
- whether the selected path is prompt-override (which bypasses perception).

If `prompt` is non-empty in video mode, pipeline can skip VLM perception depending on branch logic in `main.py`.

## 6. Config and Environment Variables

Important runtime controls:
- `VLM_API_URL`, `VLM_MODEL_NAME`
- `AUDIO_API_URL`
- `MAX_PERCEPTION_FRAMES`, `PERCEPTION_RESIZE_TO`, `PERCEPTION_CENTER_CROP`
- `MAX_VIDEO_SECONDS`
- `PROMPT_ONLY_DURATION_SEC`
- output dirs:
  - `GENERATED_AUDIO_DIR`
  - `GENERATED_VIDEO_DIR`
  - `GENERATED_TEMP_DIR`
  - `AGENT_LOG_DIR`

Operational hint: keep these aligned between shell session and `.env` to avoid confusing mixed behavior.

## 7. Frontend UX Design Decisions

Current behavior in `web/components/ui/bolt-style-chat.jsx`:
- Opaque event cards for readability.
- Sequential event queue + typewriter animation for reasoning events.
- Auto-scroll pinned to bottom with resize observer.
- Inline media rendering:
  - `<audio controls>` for audio output.
  - `<video controls>` for video output.
- Retry action shown when run fails.
- Video mode hides prompt input and uses visual perception path.

## 8. Failure Modes and Fast Triage

### 8.1 Upload fails (`TypeError: Failed to fetch`)
Likely CORS preflight issue (`OPTIONS` -> `405`).
Fix: ensure CORS middleware is enabled in `agent_ws_api.py`.

### 8.2 WebSocket fails to connect
Check:
- backend is running on expected host/port,
- Uvicorn has websocket dependency (`websockets`/`wsproto`),
- frontend `NEXT_PUBLIC_AGENT_WS_URL` is correct.

### 8.3 Video produced but no detailed timeline
Check if run entered prompt override branch; if yes, VLM timeline won’t run.

### 8.4 Loop quality poor despite retries
Inspect `generated_outputs/agent_logs/*_agent_trace.json` for:
- CLAP score trend,
- controller decisions,
- whether retries actually changed prompt semantics.

## 9. Suggested Team Workflow

### Role split
- UI owner: event rendering, timeline readability, retry UX.
- Orchestration owner: loop policy and event schema stability.
- Model-serving owner: endpoint reliability, load/offload behavior, latency.

### Contract-first development
- Treat websocket event payloads as stable contracts.
- Add/change event fields intentionally and document in one place.
- Keep backward compatibility in UI mappers where practical.

### Test strategy
- Unit-like stage tests in `scripts/`.
- Manual end-to-end with short clips (5-15s) and known acoustic cues.
- Keep one canonical sample clip + expected behavior checklist.

## 10. End-to-End Sequence Diagram (Text)

```text
UI                agent_ws_api                 FoleyOrchestrator              multi_model_api
| PUT /upload-video |                           |                              |
|------------------>| save file                |                              |
|<------------------| video_path               |                              |
| open ws /ws/foley |                           |                              |
|------------------>| start worker thread       |                              |
|                   |-------------------------->| run_pipeline                  |
|                   |                           |--/perception----------------->|
|                   |<==== stream events =======|<-----------------------------|
|                   |                           |--/execution------------------>|
|                   |                           |--/verification--------------->|
|                   |                           |--(retry loop as needed)      |
|                   |                           | stitch + report               |
|<==== run_completed + output_url =============|                              |
```
```

## 11. Runbook Commands

Backend websocket bridge:
```bash
./server/run_agent_ws_api.sh
```

Model API:
```bash
./server/run_multi_model_api.sh
```

Frontend:
```bash
cd web && npm run dev
```

Health checks:
```bash
curl http://localhost:8010/health
curl http://localhost:8000/health
```

This document should be updated whenever event schema or control flow changes.